# Chapter 24 — Production and Profiling

Part 8: Advanced Triton operations and production.

**Thread.**

- **Contract.** A Triton kernel becomes production code only when its forward, backward, and shape/dtype/device contract are all verified.
- **VJP.** The backward is the local Jacobian applied to the upstream gradient, not a second forward.
- **Measurement.** Tune, benchmark, and profile only after the correctness gates pass.
- **Chain.** PyTorch integration -> `torch.autograd.Function` -> forward/backward kernels -> gradient checks -> autotune -> benchmark -> Nsight -> roofline/occupancy -> portability.

**Ladder:** plain PyTorch reference and manual vector–Jacobian product (Method 01) → Triton forward/backward kernels inside `torch.autograd.Function` (Method 02) → an autotuned, benchmarked, and profiled production wrapper (Method 03).

**Study tip.** Write the contract first: $y=x^2$ and $\partial L/\partial x=2x\,\partial L/\partial y$. Then check one lane by hand: with $x=2$ and $h=10^{-6}$, $\frac{(2+h)^2-(2-h)^2}{2h}=4=2x$.

**Learning objectives**

- State the `torch.autograd.Function` contract: what `forward` saves, what `backward` returns, and which shapes, dtypes, and devices must match.
- Write a Triton forward kernel and its vector–Jacobian backward kernel, then gate both with `torch.testing.assert_close`.
- Use `torch.autograd.gradcheck` in `float64` to validate the analytic gradient against a finite difference.
- Choose an autotune key and a benchmark protocol (warm-up, synchronize, CUDA events, median) that produce a number you can trust.
- Read a roofline point and an occupancy limit, and keep the same kernel valid on NVIDIA and AMD backends.
- **Correctness/benchmark rule:** assert correctness before timing; a fast wrong kernel is worse than a slow correct one.

**Navigation**

- [Index](../../00_index.ipynb) · [Chapter README](README.md) · [← Previous](../ch23_transformers/23_complete.ipynb)
- **Estimated time:** 100–130 min · **Difficulty:** Advanced
- **Prerequisites:** Chapter 23 (transformer kernels), PyTorch autograd (`backward`, `requires_grad`), and Chapter 18 (1-D launch grids and tail masks).
- **Book apparatus:** [Setup](../../docs/SETUP.md) · [Notation](../../docs/NOTATION.md) · [Glossary](../../docs/GLOSSARY.md) · [API index](../../docs/API_INDEX.md) · [Formula index](../../docs/FORMULA_INDEX.md) · [Diagrams](../../docs/DIAGRAMS.md) · [Troubleshooting](../../docs/TROUBLESHOOTING.md) · [Exercise hints](../../docs/SOLUTIONS.md)

## Concept diagrams

### Host/device boundary

```text
host (Python)                         device (Triton kernel)
------------------------------        ---------------------------
check shape / dtype / device          program_id -> offsets
allocate out = empty_like(x)          mask = offsets < n
choose grid = cdiv(n, BLOCK)  ---->   load x, compute y, store y
return out  <-------------------      no allocation, no Python loop
```

- **Host.** Owns shapes, allocation, and launch geometry.
- **Device.** Owns one tile per program; no allocation and no Python loop.
- **Wrapper.** The thin contract between the two.

### Autograd graph

```text
x ---- forward kernel ----> y ---- loss L
 |                            |
 | ctx.save_for_backward(x)   | dL/dy  (upstream gradient)
 |                            v
 +---- backward kernel <--- dL/dx = 2 x * dL/dy
```

- **Backward input.** The upstream gradient already accumulated by the rest of the graph.
- **Backward output.** The local vector–Jacobian product, not the final loss gradient.

### Tuning space

```text
config c = (BLOCK, num_warps, num_stages)
        |
        +--> launch geometry: grid = cdiv(n, BLOCK)
        +--> occupancy: warps per program and registers per thread
        +--> traffic: how many tiles each program loads

c* = argmin_c median(time(c))     for a fixed runtime key (n_elements)
```

- **Config.** `BLOCK`, `num_warps`, and `num_stages` change geometry, occupancy, and traffic.
- **Key.** The winner is cached per runtime shape.

### Roofline

```text
performance
    ^
P_peak |--------*******  compute-bound
       |      **
       |    **  slope = BW
       |  **
       +------------------> arithmetic intensity AI = FLOPs / bytes
              AI* = P_peak / BW
```

- **Bound.** The slower roof sets the lower bound on time.
- **Ridge.** The point `AI = AI*` separates memory-bound from compute-bound.

### Occupancy

```text
one SM
+--------------------------------------------------+
| warp slots: 64 maximum                           |
| +--------+ +--------+ +--------+ +--------+      |
| | block  | | block  | | block  | | block  | ...  |
| | 8 warps| | 8 warps| | 8 warps| | 8 warps|      |
| +--------+ +--------+ +--------+ +--------+      |
| registers per SM: 65536 -> 4 blocks at 64 regs   |
+--------------------------------------------------+
occupancy = resident warps / maximum warps
```

- **Limiter.** Registers, shared memory, warp slots, and the block limit compete.
- **Goal.** Latency hiding, not 100% occupancy for its own sake.

In [ ]:
import math
import sys
from pathlib import Path
from typing import Callable

root = Path.cwd()
if (root / "00_Basic").exists():
    root = root / "00_Basic"
while root.name != "00_Basic" and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

import torch
import triton
import triton.language as tl
from triton.language.extra import libdevice

from ttools import DEVICE, HAS_GPU, bench, require_gpu, run_if_gpu

REAL_GPU = torch.cuda.is_available()
print("device:", DEVICE, "| GPU:", HAS_GPU, "| triton:", triton.__version__)

# Part I — Concepts

**Thread.**

- **Contract first.** 24.1–24.4 turn a kernel into a trainable op.
- **Gate second.** 24.5–24.6 turn correctness into repeatable tests.
- **Measure third.** 24.7–24.12 tune, benchmark, and explain the number.
- **Ship fourth.** 24.13 keeps the same source valid on NVIDIA and AMD.
- **Pattern.** Each subsection follows Connection -> Rule -> Formula -> Example -> Where -> Triton -> Failure -> Bridge.

## 24.1 PyTorch integration

- **Connection.** Builds on the Chapter 18 launch skeleton: a grid, a block, and a tail mask now sit behind a Python API.
- **Rule.** A Triton kernel is a host-side launcher: allocate an output with the input's shape, dtype, and device; launch on the current device; return a tensor autograd can consume.
- **Formula.** $Y=\operatorname{TritonKernel}(X)$ with $\operatorname{shape}(Y)=\operatorname{shape}(X)$, $\operatorname{dtype}(Y)=\operatorname{dtype}(X)$, and $\operatorname{device}(Y)=\operatorname{device}(X)$.
- **Example.** A contiguous `(n,)` input returns a `(n,)` output from `empty_like(x)` after `cdiv(n, BLOCK)` programs run.
- **Where.** Every fused activation, normalization, or attention kernel enters a model through this boundary; the kernel is the hotspot and the wrapper is the API.
- **Triton.** Use `torch.empty_like(x)`, `x.contiguous()`, `triton.cdiv(n, BLOCK)`, and `x.device`; never hard-code `"cuda"` because the active device may be `cuda`, `rocm`, or the CPU interpreter.
- **Failure.** Wrong device or dtype, or reading a non-contiguous tensor as contiguous, produces plausible garbage after a successful launch.
- **Bridge.** 24.2 adds the autograd registration that makes the returned tensor trainable.

## 24.2 `torch.autograd.Function`

- **Connection.** Extends 24.1: the wrapper now registers a backward instead of returning a detached tensor.
- **Rule.** Subclass `torch.autograd.Function`, define `forward(ctx, ...)` and `backward(ctx, grad_out, ...)` as `@staticmethod`s, and return exactly one gradient per forward input (or `None` for a non-tensor input).
- **Formula.** $Y=f(X)$ and $\frac{\partial L}{\partial X}=\frac{\partial L}{\partial Y}\frac{\partial Y}{\partial X}$.
- **Example.** The minimal contract:

```python
class Square(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)       # only tensors, only what backward needs
        return x * x

    @staticmethod
    def backward(ctx, grad_out):
        (x,) = ctx.saved_tensors
        return 2.0 * x * grad_out      # same shape, dtype, and device as x
```

- **Where.** This is how a fused Triton kernel becomes a trainable layer instead of a detached function.
- **Triton.** Save the input, not the output, when the derivative can be recomputed from the input; return a tuple even when there is one input; call `.contiguous()` on a `grad_out` that may be a zero-stride expanded view from `.sum()`.
- **Failure.** Forgetting `ctx.save_for_backward` (attribute access on `ctx` is not allowed for tensors), returning the wrong number of gradients, or returning a gradient with the wrong shape. Autograd raises late, often after the forward loss already looked correct.
- **Bridge.** 24.3 writes the backward as a second kernel.

## 24.3 Forward and backward kernels

- **Connection.** Builds on 24.2: the backward slot now contains a Triton kernel.
- **Rule.** The backward of an elementwise forward is another elementwise kernel with the same grid and mask; it reads the saved input and the upstream gradient, applies the local derivative, and writes the input gradient.
- **Formula.** $y_i=f(x_i)$ and $\frac{\partial L}{\partial x_i}=\frac{\partial L}{\partial y_i}f'(x_i)$; for $f(x)=x^2$, $\frac{\partial L}{\partial x_i}=2x_i g_i$ with $g_i=\frac{\partial L}{\partial y_i}$.
- **Example.** For $x=(1,2,3,4)$ and $g=(1,1,1,1)$, $dx=2x\,g=(2,4,6,8)$.
- **Where.** Fused activations, normalization layers, and the backward pass of attention all follow this pattern; the backward usually costs the same bandwidth as the forward.
- **Triton.** Reuse the same `BLOCK`, the same `mask = offs < n`, and the same tail handling; save only the tensor the derivative needs; recompute cheap values instead of storing them.
- **Failure.** Using the forward output instead of the input, dropping the tail mask, or forgetting that `grad_out` may be non-contiguous produces a gradient that is correct on most lanes and wrong on a few.
- **Bridge.** 24.4 turns this kernel into a correctness test.

## 24.4 Gradient correctness

- **Connection.** Builds on 24.3: the new backward kernel needs an independent oracle.
- **Rule.** The analytic gradient must match an independent oracle before any timing number is reported; the oracle is PyTorch autograd on the same input, or a central finite difference in `float64`.
- **Formula.** $E=\max_i\lvert g_i-\hat g_i\rvert$, and the check passes iff $E\le\operatorname{atol}+\operatorname{rtol}\max_i|\hat g_i|$.
- **Example.** $g=2x$ against $\hat g=2x$ gives $E=0$; a missing factor of two gives $E=\max_i|x_i|$.
- **Where.** This is the CI gate for a custom op; it catches a missing factor of two, a transposed index, and a mask that does not cover the tail.
- **Triton.** Use a small random input that includes negative values and a length that is not a multiple of `BLOCK`; compare with `torch.autograd` first and use `gradcheck` when the op is smooth.
- **Failure.** Checking only the forward, or testing one length that is a multiple of the block size, lets the tail bug survive both tests.
- **Bridge.** 24.5 turns the scalar error into a per-element gate.

## 24.5 `torch.testing.assert_close`

- **Connection.** Refines 24.4's scalar error into an elementwise comparison with shape, dtype, and device checks.
- **Rule.** A tolerance turns a visual comparison into a repeatable gate; `assert_close` checks every element and reports the first mismatch with both values.
- **Formula.** $\lvert a_i-e_i\rvert\le\operatorname{atol}+\operatorname{rtol}\lvert e_i\rvert$ for every $i$.
- **Example.** Typical tolerances:

| dtype | `atol` | `rtol` |
|---|---|---|
| `float32` | `1e-5` | `1e-5` |
| `float16` / `bfloat16` | `1e-2` | `1e-2` |

- **Where.** Use it for forward outputs and gradients; it is stricter than `allclose` because it also checks shape, dtype, device, and (by default) `nan` placement.
- **Triton.** Pick tolerances from the dtype; gradients are usually looser than forwards.
- **Failure.** Default tolerances on `float16` produce false failures; `.item()` before the comparison throws away shape and index information; `equal_nan=False` turns a legitimate `nan` into a mismatch.
- **Bridge.** 24.6 uses a finite difference when no autograd oracle exists.

## 24.6 `torch.autograd.gradcheck`

- **Connection.** Builds on 24.5 by replacing the autograd oracle with a finite-difference oracle in `float64`.
- **Rule.** `gradcheck` compares the analytic backward with a central finite difference of the forward in `float64`; it calls the function once per input element, so keep the tensor tiny.
- **Formula.** $\hat g_i=\frac{f(x+\varepsilon e_i)-f(x-\varepsilon e_i)}{2\varepsilon}$ and $\lVert g-\hat g\rVert\le\operatorname{atol}+\operatorname{rtol}\lVert\hat g\rVert$.
- **Example.** At $x=2$ with $h=10^{-6}$, $\frac{(2+h)^2-(2-h)^2}{2h}=4=2x$.
- **Where.** It validates the `autograd.Function` contract itself, independently of any training loss; a pass means the backward is the derivative of the forward, not just `close` on one input.
- **Triton.** Use `dtype=torch.float64`, `eps=1e-6`, `atol=1e-5`, `rtol=1e-3`, and at most a handful of elements; on a backend without practical `float64`, compare the analytic VJP with autograd in `float64` on the host.
- **Failure.** Running `gradcheck` on `float32` (the finite difference is dominated by rounding), checking a non-differentiable op (`floor`, `argmax`), or mutating the input in place inside `forward`.
- **Bridge.** 24.7 moves from correctness to choosing a launch config.

## 24.7 Triton autotuning

- **Connection.** After correctness, the first performance lever is launch geometry.
- **Rule.** `@triton.autotune` benchmarks every candidate config on the first call for a given key and caches the winner; the key must contain every runtime value that changes the best config.
- **Formula.** $c^\star=\arg\min_{c\in C}\operatorname{median}(t(c;\text{key}))$, with $c=(\texttt{BLOCK},\texttt{num\_warps},\texttt{num\_stages})$ and $\text{grid}=\lceil N/\texttt{BLOCK}\rceil$.
- **Example.** Configs `(BLOCK=128, num_warps=2)`, `(256, 4)`, `(512, 8)`; the chapter's synthetic timings choose `BLOCK=256` for the test key.
- **Where.** The best block size depends on the shape and the GPU; autotuning removes a hand-tuned constant from the source.
- **Triton.** Decorate with `@triton.autotune(configs=[...], key=["n_elements"])` and pass a grid lambda that reads `meta["BLOCK"]`; warm up once before measuring because the first call includes the tuning cost.
- **Failure.** Omitting a shape from `key` reuses the cached config for the wrong size; hard-coding the grid with a constant `BLOCK` ignores the winning config; timing the autotune search itself reports steady-state latency incorrectly.
- **Bridge.** 24.8 measures the winning config with a protocol that can be trusted.

## 24.8 Benchmarking

- **Connection.** Builds on 24.7: a tuned config is only useful if its time is measured correctly.
- **Rule.** Correctness first, then warm-up, then device synchronization, then device timers, then a robust statistic; wall-clock time around a launch measures the CPU enqueue, not the GPU work.
- **Formula.** $t_{\text{med}}=\operatorname{median}(t_1,\ldots,t_R)$ and $\mathrm{BW}_{\text{eff}}=Q/t_{\text{med}}$.
- **Example.** The worked example moves $Q=8{,}388{,}608$ bytes; at $t_{\text{med}}=5.24\ \mu s$ the effective bandwidth is about $1.6$ TB/s.
- **Where.** This is the number you compare with the roofline bound and with a competing kernel; report the median (or several quantiles), not a single run.
- **Triton.** Use `triton.testing.do_bench` or `ttools.bench`, which warms up, synchronizes, records CUDA events, and returns milliseconds per call; on a CPU-only machine the helper returns `nan` as a deliberate no-op.
- **Failure.** No warm-up (compilation and cold caches dominate), no synchronize (the timer stops before the kernel finishes), or timing a kernel that has not passed its correctness assert.
- **Bridge.** 24.9 asks why the measured kernel is slow.

## 24.9 Nsight Compute

- **Connection.** Follows 24.8: the benchmark says *which* kernel is slow; Nsight Compute says *why*.
- **Rule.** Nsight Compute reports per-kernel hardware counters; use it after a benchmark identifies the slow kernel.
- **Formula.** $t_{\text{kernel}}\approx t_{\text{memory}}+t_{\text{compute}}+t_{\text{stall}}$ and $\text{throughput}=\text{work}/t_{\text{kernel}}$.
- **Example.** DRAM throughput near peak with SM throughput low means memory-bound; SM throughput near peak means compute-bound; both low with a high stall reason means latency-bound.
- **Where.** It distinguishes the three bottleneck classes and points to the counter that explains the time.
- **Triton.** Run `ncu --set full -o report python script.py`, then read DRAM throughput, SM throughput, achieved occupancy, and the top stall reason; profile a small, fixed input and lock the clocks when comparing runs.
- **Failure.** Profiling a debug build, profiling hundreds of launches, or comparing counters across different architectures; the counter names and the peaks change.
- **Bridge.** 24.10 zooms out from one kernel to the whole timeline.

## 24.10 Nsight Systems

- **Connection.** Follows 24.9: per-kernel counters become a whole-run timeline.
- **Rule.** Nsight Systems reports kernel durations, launch gaps, memory copies, and synchronization points; it answers whether the GPU is busy, not why one kernel is slow.
- **Formula.** $T_{\text{total}}=\sum_k T_{\text{kernel},k}+T_{\text{gaps}}+T_{\text{copies}}$.
- **Example.** A timeline dominated by gaps or copies cannot be fixed by a 10% kernel improvement; a 50% idle GPU needs launch or copy work.
- **Where.** A model can spend more time in launch gaps and host–device copies than in kernels; the timeline exposes that before you optimize arithmetic.
- **Triton.** Run `nsys profile --trace=cuda,nvtx -o trace python script.py`; add NVTX ranges around the forward, backward, and optimizer step; look for gaps between launches and for copies that could be pinned or overlapped.
- **Failure.** Optimizing one kernel while the timeline is dominated by gaps or copies.
- **Bridge.** 24.11 turns the counters into a performance model.

## 24.11 Roofline analysis

- **Connection.** Builds on 24.8–24.10: the measured time is compared with a roof instead of with intuition.
- **Rule.** The attainable performance is the smaller of the compute roof and the bandwidth roof; arithmetic intensity decides which roof applies.
- **Formula.** $\mathrm{AI}=F/Q$, $t\ge\max(F/P_{\text{peak}},Q/\mathrm{BW})$, $P_{\text{attain}}\le\min(P_{\text{peak}},\mathrm{BW}\cdot\mathrm{AI})$, and the ridge is $\mathrm{AI}^\star=P_{\text{peak}}/\mathrm{BW}$.
- **Example.** For $N=2^{20}$ `float32`, a square moves $Q=8{,}388{,}608$ bytes and performs $F=1{,}048{,}576$ FLOPs, so $\mathrm{AI}=0.125$ and $\mathrm{AI}^\star=12.5$; the kernel is memory-bound with $t_{\text{mem}}=5.24\ \mu s$ and $t_{\text{compute}}=0.052\ \mu s$.
- **Where.** If $\mathrm{AI}\lt \mathrm{AI}^\star$, reduce bytes (fusion, narrower dtype, fewer intermediate tensors); if $\mathrm{AI}\gt \mathrm{AI}^\star$, reduce FLOPs or use tensor cores.
- **Triton.** Count every byte that crosses the HBM boundary, each input read and each output write; use one unit system (decimal GB/s with decimal bytes, or GiB/s with binary bytes) throughout.
- **Failure.** Unit errors (GB versus GiB, FLOP versus GFLOP), counting cache hits as HBM traffic, or forgetting the output write shifts the point across the ridge.
- **Bridge.** 24.12 explains whether the SM can hide the memory latency behind resident warps.

## 24.12 Occupancy analysis

- **Connection.** Follows 24.11: the roofline says memory-bound; occupancy says whether the SM can hide that memory latency.
- **Rule.** Occupancy is the fraction of the SM's warp slots that are resident; it is limited by registers, shared memory, warp slots, and the hardware block limit.
- **Formula.**

$$
\begin{aligned}
\text{warps/block}&=\frac{\text{threads/block}}{w},\\
B_{\text{reg}}&=\left\lfloor\frac{R_{\text{SM}}}{\text{regs/thread}\cdot\text{threads/block}}\right\rfloor,\\
B_{\text{smem}}&=\left\lfloor\frac{S_{\text{SM}}}{\text{smem/block}}\right\rfloor,\\
B&=\min(B_{\text{reg}},B_{\text{smem}},B_{\text{warp}},B_{\text{hw}}),\\
\text{occupancy}&=\frac{B\cdot\text{warps/block}}{W_{\max}}.
\end{aligned}
$$

- **Example.** With 64 registers/thread, 256 threads/block, $w=32$, $R_{\text{SM}}=65{,}536$, and $W_{\max}=64$: warps/block $=8$, $B_{\text{reg}}=4$, $B_{\text{warp}}=8$, so $B=4$ and occupancy $=4\cdot8/64=0.5$.
- **Where.** Occupancy is a latency-hiding budget: more resident warps hide memory latency, while too many registers per thread does the opposite and may also spill.
- **Triton.** `num_warps` sets warps per program; check achieved occupancy in Nsight Compute and do not infer it from `num_warps` alone.
- **Failure.** Assuming 100% occupancy is always best, ignoring register spills, or using the NVIDIA warp size on AMD; a kernel can be faster at 50% occupancy if it has more registers per thread and fewer spills.
- **Bridge.** 24.13 makes the warp size explicit so the same formula works on AMD.

## 24.13 Hardware portability

- **Connection.** Builds on 24.12, which used $w=32$; portability makes $w$ a backend fact instead of a hidden constant.
- **Rule.** The same Triton source should compile for NVIDIA and AMD; keep the grid and masks independent of the warp size, prefer portable `tl.*` operations, and isolate backend-specific intrinsics.
- **Formula.** $\text{threads/program}=\texttt{num\_warps}\cdot w$, with $w=32$ on NVIDIA and $w=64$ on AMD.
- **Example.** `num_warps=4` gives 128 threads/program on NVIDIA and 256 on AMD, while the grid still uses `cdiv(N, BLOCK)`.
- **Where.** A production kernel ships to more than one accelerator; tile sizes, dtype support, and `libdevice` coverage differ between backends.
- **Triton.** Use `triton.cdiv`, power-of-two block sizes, masks for the tail, and `tl.maximum`/`tl.minimum`/`tl.exp2` rather than inline PTX; guard `libdevice` calls and test fp16/bf16 on each target.
- **Failure.** Hard-coding `32` threads per warp, using inline assembly, assuming fp8 support, or calling a `libdevice` symbol that one backend does not implement compiles on one vendor and fails on the other.
- **Bridge.** The chapter map and worked example turn these rules into one path.
- **Where this leads.** Part II starts with the scalar reference that every later kernel is checked against, and the worked example pins the numbers before any code runs.

## Chapter map

- **Reading.** Rows 24.1–24.13 follow Part I; the capstone ties the correctness, tuning, and profiling gates together.

| § | Topic | Rule/formula | Check | Demo |
|---|---|---|---|---|
| **24.1** | PyTorch integration | $Y=K(X)$ preserves shape, dtype, device | output metadata | manual square |
| **24.2** | `torch.autograd.Function` | $\partial L/\partial X=(\partial L/\partial Y)(\partial Y/\partial X)$ | contract + `gradcheck` | `ManualSquare` |
| **24.3** | Forward/backward kernels | $dx_i=2x_i g_i$ | tail and non-contiguous input | `TritonSquare` |
| **24.4** | Gradient correctness | $E\le\mathrm{atol}+\mathrm{rtol}\max\lvert\hat g\rvert$ | autograd oracle | `test_triton_forward_backward` |
| **24.5** | `assert_close` | $\lvert a_i-e_i\rvert\le\mathrm{atol}+\mathrm{rtol}\lvert e_i\rvert$ | accepting and rejecting cases | tolerance gate |
| **24.6** | `gradcheck` | central difference in `float64` | `gradcheck` returns true | `test_contract_gradcheck` |
| **24.7** | Autotuning | $c^\star=\arg\min_c \operatorname{median}(t_c)$ | key contains the shape | `tuned_scale` |
| **24.8** | Benchmarking | $t_{\text{med}}$, $\mathrm{BW}_{\text{eff}}=Q/t$ | correctness before timing | `test_benchmark` |
| **24.9** | Nsight Compute | counters: memory, compute, stalls | one kernel at a time | profiler checklist |
| **24.10** | Nsight Systems | $T=\sum T_k+T_{\text{gaps}}+T_{\text{copies}}$ | timeline has no gaps | NVTX ranges |
| **24.11** | Roofline | $t\ge\max(F/P_{\text{peak}},Q/\mathrm{BW})$ | intensity versus ridge | `roofline` worksheet |
| **24.12** | Occupancy | $B=\min(B_{\text{reg}},B_{\text{smem}},B_{\text{warp}},B_{\text{hw}})$ | limiter and fraction | `occupancy` worksheet |
| **24.13** | Portability | $\text{threads}=\texttt{num\_warps}\cdot w$ | NVIDIA $w=32$, AMD $w=64$ | portability asserts |
| **★** | Capstone | correctness → gradient → tune → benchmark → profile | every gate green | production checklist |

## Worked numerical example

- **Square.** The tiny kernel is the same map that Part III wraps in `torch.autograd.Function`: $y=x^2$, $x=(1,2,3,4)$, $y=(1,4,9,16)$.
- **Backward.** With an upstream gradient of ones, the manual backward is $\partial L/\partial x_i=2x_i=(2,4,6,8)$.
- **Gradient check.** At $x=2$ and $h=10^{-6}$, $\frac{(2+h)^2-(2-h)^2}{2h}=\frac{4h}{2h}=4=2x$.
- **Tolerance.** The analytic and numerical values differ only by rounding, so the check passes with $\lvert g-\hat g\rvert\le\operatorname{atol}+\operatorname{rtol}\lvert\hat g\rvert=10^{-5}+10^{-3}\cdot4=4.01\times10^{-3}$.
- **Roofline bytes.** For $N=2^{20}$ `float32`, a square reads one array and writes one array: $Q=2\cdot4\cdot N=8{,}388{,}608$ bytes and $F=N=1{,}048{,}576$ FLOPs.
- **Roofline intensity.** $\mathrm{AI}=F/Q=0.125$ and $\mathrm{AI}^\star=P_{\text{peak}}/\mathrm{BW}=20\times10^{12}/1.6\times10^{12}=12.5$.
- **Roofline bound.** Since $0.125\ll12.5$, the kernel is memory-bound: $t_{\text{mem}}=Q/\mathrm{BW}=5.24\ \mu s$, $t_{\text{compute}}=F/P_{\text{peak}}=0.052\ \mu s$, and $t\ge5.24\ \mu s$.
- **Occupancy.** With 64 registers/thread, 256 threads/block, warp size 32, register file 65,536 registers/SM, and $W_{\max}=64$ warps/SM: $\text{warps/block}=8$, $B_{\text{reg}}=\lfloor65{,}536/(64\cdot256)\rfloor=4$, and $B_{\text{warp}}=\lfloor64/8\rfloor=8$.
- **Occupancy result.** With no shared memory and a hardware block limit above 4, $B=4$ and $\text{occupancy}=4\cdot8/64=0.5$.
- **Check.** The next cell asserts every number above without printing a table.

In [ ]:
# Worked example: forward, manual VJP, gradient tolerance, roofline, occupancy.
x = torch.tensor([1.0, 2.0, 3.0, 4.0])
y = x * x
g = torch.ones_like(x)
dx = 2.0 * x * g

assert y.tolist() == [1.0, 4.0, 9.0, 16.0]
assert dx.tolist() == [2.0, 4.0, 6.0, 8.0]

h = 1e-6
numeric = ((2.0 + h) ** 2 - (2.0 - h) ** 2) / (2.0 * h)
assert abs(numeric - 4.0) < 1e-5
atol, rtol = 1e-5, 1e-3
assert abs(dx[1].item() - numeric) <= atol + rtol * abs(numeric)

N = 1 << 20
bytes_moved = 2 * 4 * N
flops = N
intensity = flops / bytes_moved
peak_flops, bandwidth = 20e12, 1.6e12
ridge = peak_flops / bandwidth
t_memory = bytes_moved / bandwidth
t_compute = flops / peak_flops

assert bytes_moved == 8_388_608
assert flops == 1_048_576
assert abs(intensity - 0.125) < 1e-15
assert abs(ridge - 12.5) < 1e-12
assert abs(t_memory - 5.24288e-6) < 1e-12
assert abs(t_compute - 5.24288e-8) < 1e-12
assert intensity < ridge and t_memory > t_compute

regs_per_thread, threads_per_block = 64, 256
warp_size, regs_per_sm, max_warps = 32, 65_536, 64
warps_per_block = threads_per_block // warp_size
blocks_reg = regs_per_sm // (regs_per_thread * threads_per_block)
blocks_warp = max_warps // warps_per_block
blocks = min(blocks_reg, blocks_warp, 32)
active_warps = blocks * warps_per_block
occupancy = active_warps / max_warps

assert warps_per_block == 8
assert blocks_reg == 4 and blocks_warp == 8 and blocks == 4
assert active_warps == 32
assert occupancy == 0.5

# Part II — Method 01: the reference and the manual VJP

- **Connection.** Method 01 follows Part I's contract: these scalar maps are the oracle for every later kernel.
- **Thread.** One Python iteration per element; slow, but exact.
- **Forward.** $\texttt{manual\_square\_forward}(x)_i=x_i^2$.
- **Backward.** $\texttt{manual\_square\_backward}(x,g)_i=2x_i g_i$.
- **Why separate.** The split mirrors `torch.autograd.Function`: the forward saves state, and the backward receives the upstream gradient.

In [ ]:
def manual_square_forward(x: torch.Tensor) -> torch.Tensor:
    '''Method 01 forward: one scalar square per element.'''
    x = torch.as_tensor(x)
    out = torch.empty_like(x)
    xf, of = x.reshape(-1), out.reshape(-1)
    for i in range(xf.numel()):
        of[i] = xf[i] * xf[i]
    return out

In [ ]:
def manual_square_backward(x: torch.Tensor, grad_out: torch.Tensor) -> torch.Tensor:
    '''Method 01 backward: one scalar VJP per element.'''
    x = torch.as_tensor(x)
    grad_out = torch.as_tensor(grad_out)
    assert x.shape == grad_out.shape
    out = torch.empty_like(x)
    xf, gf, of = x.reshape(-1), grad_out.reshape(-1), out.reshape(-1)
    for i in range(xf.numel()):
        of[i] = 2.0 * xf[i] * gf[i]
    return out

In [ ]:
def test_method01() -> None:
    '''CPU: the scalar definition, the shape rule, and the autograd oracle.'''
    x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
    assert manual_square_forward(x).tolist() == [4.0, 1.0, 0.0, 1.0, 4.0]

    grad = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
    manual = manual_square_backward(x, grad)
    assert manual.shape == x.shape and manual.dtype == x.dtype
    assert manual.tolist() == [-4.0, -4.0, 0.0, 8.0, 20.0]

    xr = x.clone().requires_grad_(True)
    (xr * xr * grad).sum().backward()
    torch.testing.assert_close(xr.grad, manual)

    # Zero and negative lanes are part of the contract, not an afterthought.
    assert manual_square_backward(x, torch.zeros_like(x)).tolist() == [0.0] * 5


test_method01()
print("method 01: forward and manual VJP match the definition and autograd")

### The vector–Jacobian product

- **Connection.** Builds on 24.3 and the Method 01 backward.
- **Rule.** The backward is the transpose of the local Jacobian applied to the upstream gradient.
- **Formula.** $J_{ij}=\partial y_i/\partial x_j$ and $\partial L/\partial x_j=\sum_i(\partial L/\partial y_i)J_{ij}=(J^\top \partial L/\partial y)_j$.
- **Example.** For the elementwise square, $J$ is diagonal with entries $2x_i$, so the VJP is the elementwise product $2x_i g_i$.
- **Shape.** $\operatorname{shape}(dx)=\operatorname{shape}(x)$, $\operatorname{dtype}(dx)=\operatorname{dtype}(x)$, and $\operatorname{device}(dx)=\operatorname{device}(x)$.
- **Broadcast.** If the forward broadcast a size-1 axis, the backward would sum over that axis; the square does not broadcast, so no reduction appears in `backward`.

# Part III — Method 02: Triton kernels inside `torch.autograd.Function`

- **Connection.** Method 02 keeps Method 01's mathematics and moves the two maps to the device.
- **Forward.** The forward kernel writes $x^2$.
- **Backward.** The backward kernel writes $2xg$.
- **Skeleton.** Both use offsets $\text{pid}\cdot B+\{0,\ldots,B-1\}$, mask $\text{offs}\lt N$, and grid $P=\lceil N/B\rceil$.

In [ ]:
@triton.jit
def square_fwd_kernel(x_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    offs = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, x * x, mask=mask)


@triton.jit
def square_bwd_kernel(x_ptr, g_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    offs = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    g = tl.load(g_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, 2.0 * x * g, mask=mask)

In [ ]:
class TritonSquare(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        x = x.contiguous()
        out = torch.empty_like(x)
        n = x.numel()
        square_fwd_kernel[(triton.cdiv(n, 256),)](x, out, n, BLOCK=256)
        ctx.save_for_backward(x)
        return out

    @staticmethod
    def backward(ctx, grad_out):
        (x,) = ctx.saved_tensors
        grad_out = grad_out.contiguous()
        out = torch.empty_like(x)
        n = x.numel()
        square_bwd_kernel[(triton.cdiv(n, 256),)](x, grad_out, out, n, BLOCK=256)
        return out


def triton_square(x: torch.Tensor) -> torch.Tensor:
    return TritonSquare.apply(x)

### What the wrapper saves and returns

- **Connection.** Builds on 24.2 and the Method 02 kernels.
- **Save.** `forward` saves the contiguous input, not the output, because $f'(x)=2x$ can be recomputed from $x$ alone.
- **Backward.** `backward` receives `grad_out`, makes it contiguous, launches the gradient kernel, and returns one tensor.
- **Invariants.** $\operatorname{shape}(dx)=\operatorname{shape}(x)$, $\operatorname{dtype}(dx)=\operatorname{dtype}(x)$, and $\operatorname{device}(dx)=\operatorname{device}(x)$.
- **Why contiguous.** `.sum()` can hand the backward a zero-stride expanded view; a kernel that assumes contiguous addresses would read the wrong lane.
- **Next.** The next cell checks the contract without a kernel, then the GPU tests check the kernel version.

In [ ]:
class ManualSquare(torch.autograd.Function):
    '''The same contract without a kernel: the CPU-safe gradcheck target.'''
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x * x

    @staticmethod
    def backward(ctx, grad_out):
        (x,) = ctx.saved_tensors
        return 2.0 * x * grad_out


def test_contract_gradcheck() -> None:
    x = torch.tensor([0.5, -1.5, 2.0, 0.25], dtype=torch.float64, requires_grad=True)
    assert torch.autograd.gradcheck(
        ManualSquare.apply, (x,), eps=1e-6, atol=1e-5, rtol=1e-3
    )
    y = ManualSquare.apply(x)
    assert y.shape == x.shape and y.dtype == torch.float64
    y.sum().backward()
    torch.testing.assert_close(x.grad, 2.0 * x.detach())


test_contract_gradcheck()
print("autograd contract: float64 gradcheck passes; shape and dtype are preserved")

In [ ]:
def test_assert_close_gate() -> None:
    expected = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float32)
    torch.testing.assert_close(
        expected + 1e-7, expected, atol=1e-5, rtol=1e-5
    )

    # A 1% error must be rejected by a 1e-3 tolerance.
    try:
        torch.testing.assert_close(
            expected * 1.01, expected, atol=1e-3, rtol=1e-3
        )
    except AssertionError:
        pass
    else:
        raise AssertionError("assert_close accepted a 1% error")

    # fp16 needs a looser tolerance than fp32.
    half = expected.to(torch.float16)
    torch.testing.assert_close(
        half + torch.tensor(2e-3, dtype=torch.float16), half, atol=1e-2, rtol=1e-2
    )

    grad = torch.tensor([2.0, 4.0, 6.0])
    torch.testing.assert_close(grad, 2.0 * expected, atol=1e-5, rtol=1e-5)


test_assert_close_gate()
print("assert_close: rejects 1% error, accepts 1e-7, and uses dtype-aware tolerances")

In [ ]:
def test_triton_forward_backward() -> None:
    require_gpu("TritonSquare forward/backward")
    torch.manual_seed(0)
    x = torch.randn(1003, device=DEVICE, requires_grad=True)
    y = triton_square(x)
    torch.testing.assert_close(y, x * x)

    w = torch.randn_like(y)
    (y * w).sum().backward()
    torch.testing.assert_close(x.grad, 2.0 * x.detach() * w)

    # .sum() produces a zero-stride expanded gradient; .contiguous() handles it.
    x2 = torch.randn(17, device=DEVICE, requires_grad=True)
    triton_square(x2).sum().backward()
    torch.testing.assert_close(x2.grad, 2.0 * x2.detach())


def test_triton_tail_and_noncontiguous() -> None:
    require_gpu("TritonSquare tail and non-contiguous")
    x = torch.randn(1003, device=DEVICE)
    torch.testing.assert_close(triton_square(x), x * x)

    base = torch.randn(1003, 7, device=DEVICE)
    view = base.t()
    assert view.shape == (7, 1003) and not view.is_contiguous()
    torch.testing.assert_close(triton_square(view), view * view)


def test_triton_gradcheck() -> None:
    require_gpu("TritonSquare gradcheck")
    x = torch.tensor(
        [0.5, -1.5, 2.0, 0.25], device=DEVICE, dtype=torch.float64,
        requires_grad=True,
    )
    if DEVICE.type == "cuda":
        assert torch.autograd.gradcheck(
            lambda t: triton_square(t), (x,), eps=1e-6, atol=1e-5, rtol=1e-3
        )
    else:
        # CPU interpreter: the CPU-safe analytic path in float64.
        w = torch.tensor([1.0, -2.0, 0.5, 3.0], device=DEVICE, dtype=torch.float64)
        (triton_square(x) * w).sum().backward()
        torch.testing.assert_close(x.grad, 2.0 * x.detach() * w)

# Part IV — Method 03: tune, benchmark, and profile

- **Connection.** Method 03 wraps the verified Method 02 kernel in the production loop.
- **Tune.** Autotune the launch geometry per runtime key.
- **Measure.** Benchmark with a protocol that can be trusted.
- **Explain.** Compare the result with a roofline/occupancy model.
- **Skip.** GPU-only paths are skipped on a CPU; the autotuner stays behind a real-driver check because it benchmarks candidate configs by launching them.

In [ ]:
@triton.autotune(
    configs=[
        triton.Config({"BLOCK": 128}, num_warps=2),
        triton.Config({"BLOCK": 256}, num_warps=4),
        triton.Config({"BLOCK": 512}, num_warps=8),
    ],
    key=["n_elements"],
)
@triton.jit
def tuned_scale_kernel(x_ptr, out_ptr, n_elements, scale, BLOCK: tl.constexpr):
    offs = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, x * scale, mask=mask)


def tuned_scale(x: torch.Tensor, scale: float = 2.0) -> torch.Tensor:
    x = x.contiguous()
    out = torch.empty_like(x)
    grid = lambda meta: (triton.cdiv(x.numel(), meta["BLOCK"]),)
    tuned_scale_kernel[grid](x, out, x.numel(), scale)
    return out

In [ ]:
CONFIGS = [
    {"BLOCK": 128, "num_warps": 2},
    {"BLOCK": 256, "num_warps": 4},
    {"BLOCK": 512, "num_warps": 8},
]


def choose_config(timings: dict[int, float], configs: list[dict]) -> dict:
    '''The autotuner's decision, isolated as a pure function.'''
    return min(configs, key=lambda c: timings[c["BLOCK"]])


def test_autotune_selection() -> None:
    timings = {128: 0.040, 256: 0.032, 512: 0.051}
    best = choose_config(timings, CONFIGS)
    assert best["BLOCK"] == 256 and best["num_warps"] == 4

    key = ["n_elements"]
    assert "n_elements" in key          # the runtime shape is part of the cache key
    assert all(c["BLOCK"] & (c["BLOCK"] - 1) == 0 for c in CONFIGS)

    # The grid must be derived from the winning BLOCK, not a constant.
    n = 10_000
    for c in CONFIGS:
        assert triton.cdiv(n, c["BLOCK"]) == (n + c["BLOCK"] - 1) // c["BLOCK"]


def test_tuned_scale() -> None:
    if not REAL_GPU:
        return                          # autotune needs a live driver; CPU smoke skips
    x = torch.randn(10_000, device="cuda")
    torch.testing.assert_close(tuned_scale(x, 2.5), x * 2.5)


test_autotune_selection()
print("autotune: synthetic timings choose BLOCK=256; the shape is part of the key")

In [ ]:
def time_ms(fn: Callable[[], None], warmup: int = 10, rep: int = 50) -> float:
    '''Device time via ttools.bench; NaN is the CPU no-op.'''
    if not REAL_GPU:
        return float("nan")
    return bench(fn, warmup=warmup, rep=rep)


@triton.jit
def bench_scale_kernel(x_ptr, out_ptr, n_elements, scale, BLOCK: tl.constexpr):
    offs = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, x * scale, mask=mask)


def bench_scale(x: torch.Tensor, scale: float = 2.0, BLOCK: int = 1024) -> torch.Tensor:
    x = x.contiguous()
    out = torch.empty_like(x)
    bench_scale_kernel[(triton.cdiv(x.numel(), BLOCK),)](
        x, out, x.numel(), scale, BLOCK=BLOCK
    )
    return out

In [ ]:
def test_benchmark() -> None:
    require_gpu("benchmark")
    x = torch.randn(1 << 20, device=DEVICE)

    # The correctness gate comes before any timing number.
    torch.testing.assert_close(bench_scale(x), 2.0 * x)

    ms = time_ms(lambda: bench_scale(x))
    if REAL_GPU:
        assert ms > 0.0
    else:
        assert math.isnan(ms)           # CPU interpreter: the timer is a no-op

    bytes_moved = 2 * x.numel() * x.element_size()
    assert bytes_moved == 8_388_608
    print("benchmark: correctness asserted first; timer is a no-op without a GPU")

In [ ]:
def roofline(n, bytes_per_elem, flops_per_elem, peak_flops, bandwidth):
    bytes_moved = 2 * bytes_per_elem * n          # one read + one write
    flops = flops_per_elem * n
    intensity = flops / bytes_moved
    t_memory = bytes_moved / bandwidth
    t_compute = flops / peak_flops
    return {
        "bytes": bytes_moved,
        "flops": flops,
        "intensity": intensity,
        "ridge": peak_flops / bandwidth,
        "t_memory": t_memory,
        "t_compute": t_compute,
        "bound": "memory" if t_memory >= t_compute else "compute",
    }


def occupancy(regs_per_thread, threads_per_block, regs_per_sm=65_536,
              max_warps_per_sm=64, warp_size=32, smem_per_block=0,
              smem_per_sm=0, max_blocks_per_sm=32):
    warps_per_block = threads_per_block // warp_size
    blocks_reg = regs_per_sm // (regs_per_thread * threads_per_block)
    blocks_warp = max_warps_per_sm // warps_per_block
    blocks_smem = (smem_per_sm // smem_per_block) if smem_per_block else max_blocks_per_sm
    blocks = min(blocks_reg, blocks_warp, blocks_smem, max_blocks_per_sm)
    active_warps = blocks * warps_per_block
    return {
        "warps_per_block": warps_per_block,
        "blocks_reg": blocks_reg,
        "blocks_warp": blocks_warp,
        "blocks": blocks,
        "active_warps": active_warps,
        "occupancy": active_warps / max_warps_per_sm,
    }


def test_profiling_worksheet() -> None:
    r = roofline(1 << 20, 4, 1, 20e12, 1.6e12)
    assert r["bytes"] == 8_388_608 and r["flops"] == 1_048_576
    assert abs(r["intensity"] - 0.125) < 1e-15
    assert abs(r["ridge"] - 12.5) < 1e-12
    assert r["bound"] == "memory"

    o = occupancy(64, 256)
    assert o["warps_per_block"] == 8 and o["blocks"] == 4
    assert o["active_warps"] == 32 and o["occupancy"] == 0.5

    # Halving the registers doubles the resident blocks and reaches 100%.
    o2 = occupancy(32, 256)
    assert o2["blocks"] == 8 and o2["occupancy"] == 1.0

    # A 128-thread block at 64 registers is limited by the register file too.
    o3 = occupancy(64, 128)
    assert o3["warps_per_block"] == 4 and o3["blocks"] == 8
    assert o3["occupancy"] == 0.5


test_profiling_worksheet()
print("roofline/occupancy: 0.125 FLOP/byte is memory-bound; 64 regs/thread gives 50% occupancy")

## Production checklist

- **Order.** Run the gates in this order; each is cheaper than the next, and a later gate cannot repair an earlier failure.
- **Rule.** Do not report gate 5 until gates 1–3 are green; a fast kernel with a wrong backward is a regression, not an optimization.

| Gate | Question | Tool | Pass condition |
|---|---|---|---|
| 1. Forward | Does the kernel compute the right values? | `torch.testing.assert_close` | every element within tolerance |
| 2. Backward | Is the VJP the derivative of the forward? | autograd oracle, then `gradcheck` | `gradcheck` returns true |
| 3. Shape contract | Are tail, dtype, device, and contiguity preserved? | `assert`, odd $N$, transposed view | no exception |
| 4. Tune | Which config wins for this key? | `@triton.autotune` | same output after tuning |
| 5. Benchmark | How long does the winning config take? | warm-up + events + median | stable, correctness still green |
| 6. Profile | Is the kernel memory-, compute-, or latency-bound? | `ncu`, `nsys` | counters agree with the roofline model |

# Part V — Hardware portability

- **Thread.** The source is the same; the backend is not.
- **First differences.** Warp size and available dtypes change first.

| Feature | NVIDIA | AMD | Portable rule |
|---|---|---|---|
| warp size $w$ | 32 | 64 | never hard-code 32; derive threads from `num_warps * w` |
| fp32 / fp16 / bf16 | yes | yes | test all three on the target |
| fp8 | recent GPUs | backend-dependent | feature-gate it; do not assume |
| `libdevice` | broad coverage | subset/naming differences | prefer `tl.*`; guard backend-specific calls |
| tensor cores | `tl.dot` shapes differ | matrix-core shapes differ | autotune tile shapes per target |
| inline PTX | yes | no | keep it out of portable kernels |

- **Safe pattern.** Power-of-two `BLOCK`, `triton.cdiv` for the grid, masks on every load and store, `tl.maximum`/`tl.minimum`/`tl.exp2` for arithmetic, and a `try`/fallback around any `libdevice` symbol.
- **Result.** A kernel that follows those rules compiles on both vendors and still passes the same `assert_close` gate.

In [ ]:
NVIDIA_WARP, AMD_WARP = 32, 64
for warp in (NVIDIA_WARP, AMD_WARP):
    for num_warps in (2, 4, 8):
        threads = num_warps * warp
        assert threads % warp == 0
        assert threads in (64, 128, 256, 512)

# The grid is independent of the warp size and covers every tail.
for n in (1, 255, 256, 257, 1003, 1 << 20):
    assert triton.cdiv(n, 256) * 256 >= n
    assert triton.cdiv(n, 256) == (n + 255) // 256

PORTABLE_DTYPES = {torch.float32, torch.float16, torch.bfloat16}
assert torch.float32 in PORTABLE_DTYPES
assert torch.float16 in PORTABLE_DTYPES
assert torch.bfloat16 in PORTABLE_DTYPES

# libdevice exists, but portable kernels should not depend on a vendor-only name.
assert hasattr(libdevice, "pow")
assert callable(tl.maximum) and callable(tl.exp2)
print("portability: warp-size-aware grid, portable dtypes, and libdevice guarded")

# Pitfalls, checks, and summary

## Pitfalls

- **Pre-flight.** Use the table as a check before timing any new kernel.
- **Pattern.** Every row is a contract violation that a test in this chapter catches.

| Pitfall | What happens | Fix |
|---|---|---|
| Missing `ctx.save_for_backward(x)` | `backward` has no input; autograd raises or the derivative is wrong | save only the tensors the backward needs |
| Wrong gradient shape | autograd rejects the return or silently broadcasts | return the same shape, dtype, and device as the forward input |
| `gradcheck` on `float32` | finite differences are dominated by rounding | use `float64`, tiny inputs, `eps=1e-6` |
| Benchmark without warm-up | compilation and cold caches dominate the number | warm up, synchronize, time with CUDA events |
| Benchmark without synchronize | the timer stops before the kernel finishes | synchronize before and after the timed region |
| Autotune key missing a shape | a config tuned for one size is reused for another | put every runtime shape in `key` |
| Grid hard-codes `BLOCK` | the winning config changes `BLOCK`, but the grid does not | use `lambda meta: (cdiv(n, meta["BLOCK"]),)` |
| Roofline unit error | GB vs GiB, FLOP vs GFLOP, or a missing write | fix one unit system and count every HBM byte |
| Occupancy assumption | 100% is assumed without checking registers/spills | compute the limiter and read achieved occupancy in `ncu` |
| Non-contiguous `grad_out` | the backward kernel reads the wrong addresses | `.contiguous()` before launch |
| Hard-coded warp size | the kernel is wrong on AMD | derive threads from `num_warps * warp_size` |
| Profiling the wrong build | counters do not represent the production kernel | profile the release build with fixed clocks |

## Self-check

1. What must `forward` save, and what must `backward` return for a one-input function?
2. Why does `gradcheck` need `float64`, and what does a failure tell you?
3. Write the central-difference check for $y=x^2$ at $x=2$ with $h=10^{-6}$.
4. Why is `assert_close` stricter than `allclose`?
5. What happens if the autotune key omits `n_elements`?
6. Why must a benchmark synchronize before stopping the timer?
7. For $N=2^{20}$ `float32`, compute $Q$, $F$, $\mathrm{AI}$, and the bound.
8. With 64 registers/thread and 256 threads/block, what limits occupancy and what is the fraction?
9. Name two portability differences between NVIDIA and AMD that change a kernel launch.
10. Why is a fast wrong kernel worse than a slow correct one?

## Summary

- **Contract.** A Triton kernel becomes a trainable op through `torch.autograd.Function`: save the input, return the VJP, match shape/dtype/device.
- **Backward.** The backward is another elementwise kernel with the same grid and mask; tail and non-contiguous gradients are part of the contract.
- **Gate.** `assert_close` is the per-element gate; `gradcheck` is the finite-difference gate in `float64`; both run before timing.
- **Tune.** Autotuning chooses a config per runtime key; benchmarking uses warm-up, synchronization, device timers, and a median.
- **Explain.** The roofline decides memory versus compute; occupancy decides latency hiding; neither replaces a measured number.
- **Ship.** Portability is source discipline: no hard-coded warp size, no vendor-only intrinsics in the common path, and the same correctness gate everywhere.

**Where this leads.**

- **Next step.** Carry the same correctness-then-measurement loop into any custom kernel you ship.
- **Production loop.** Integration, autograd, gradient checks, tuning, benchmarking, profiling, and portability are one pipeline, not seven unrelated topics.

## Exercises

1. **Contract audit.** Add a `scale` argument to `TritonSquare` and return the correct gradient for both inputs. Which gradients are `None`?
2. **Tail hunt.** Change `BLOCK` from 256 to 64 and test $N=1003$. Which lane would be wrong if the backward forgot its mask?
3. **Gradcheck tolerances.** Run `gradcheck` with `eps=1e-4`, `1e-6`, and `1e-8` on `ManualSquare`. Explain the U-shaped error curve.
4. **Non-contiguous gradient.** Feed a transposed tensor and a zero-stride expanded gradient. Which `.contiguous()` calls are required, and why?
5. **Autotune key.** Add a `dtype` or `n_elements` component to the key and show what goes wrong when it is missing.
6. **Benchmark protocol.** Time `bench_scale` with and without warm-up and with and without `synchronize`. Report the three numbers and explain the gap.
7. **Roofline.** Compute the roofline point for a fused $y=a x+b$ kernel with $N=2^{20}$ `float32`. How many bytes and FLOPs, and which roof binds?
8. **Occupancy.** Vary registers/thread from 16 to 128 at 256 threads/block. At what value does the register file become the limiter?
9. **Portability.** Rewrite the grid so the same source works with warp size 32 and 64. What changes, and what must not change?
10. **Profiling.** Run `ncu` on `bench_scale` and `nsys` on the forward/backward pair. Which one answers "why is this kernel slow", and which answers "where did the time go"?

In [ ]:
print("CPU checks")
test_method01()
test_contract_gradcheck()
test_assert_close_gate()
test_autotune_selection()
test_profiling_worksheet()

print("\nGPU/interpreter checks (SKIP without CUDA/ROCm)")
for fn in (
    test_triton_forward_backward,
    test_triton_tail_and_noncontiguous,
    test_triton_gradcheck,
    test_tuned_scale,
    test_benchmark,
):
    run_if_gpu(fn)

print("\ndone: correctness gates passed; GPU checks ran or printed SKIP")